In [16]:
from pathlib import Path
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [17]:
# local embedding model import

MODEL_PATH = Path.home() / "rag-learning-practice" / "models" / "all-MiniLM-L6-v2"

# Local embedding model setup
embeddings = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4882.22it/s]


In [18]:
# প্রতিটি Document-এ page_content থাকে মূল text, আর metadata-তে extra information
documents = [
    Document(
        page_content="LangChain is a framework for building applications with large language models.",
        metadata={"source": "langchain_intro", "topic": "langchain"}
    ),
    Document(
        page_content="Retrievers help applications find relevant documents for a user query.",
        metadata={"source": "retriever_intro", "topic": "retriever"}
    ),
    Document(
        page_content="Chroma is a vector database used to store and search embeddings.",
        metadata={"source": "chroma_intro", "topic": "chroma"}
    ),
    Document(
        page_content="FAISS is a library for efficient similarity search and clustering of dense vectors.",
        metadata={"source": "faiss_intro", "topic": "faiss"}
    ),
]

In [19]:
# chroma_db vector store create korci

vector_store = Chroma(
    collection_name="rag_learning",
    embedding_function=embeddings,
    persist_directory="my_chroma_db"
)

In [20]:
# ৪. Document gulo vector store ea add korchi

vector_store.add_documents(documents)  # add_documents() every text- ke embedding ea convert kore Chroma te store kore

['1acb1a39-264e-4801-9a83-4595ba7a2602',
 '0a79d59e-422b-4089-b4ae-38c9fe9e2196',
 '45bd61f0-8be4-45c5-8175-9d0b2e2653d0',
 'b0623f4f-f663-449c-95c4-07a2b8b4dd00']

In [21]:
# ৫. Vector store-কে retriever-এ convert করছি

retriever = vector_store.as_retriever(
    search_type="similarity",  # search_type="similarity" মানে query-র সাথে সবচেয়ে similar document খুঁজবে
    search_kwargs={"k": 2}     # k=2 মানে top 2 document ফেরত দিবে
)

In [22]:
# ৬. Query দিয়ে retrieval test করছি
query = "What is a retriever in LangChain?"

In [23]:
# invoke() হলো retriever-এর main method. এটি query নেয় এবং relevant Document list return করে

results = retriever.invoke(query)

In [24]:
# ৭. Result print করছি

print("Query:", query)
print("-" * 50)

for i, doc in enumerate(results, start=1):
    print(f"Result {i}:")
    print(doc.page_content)
    print("Metadata:", doc.metadata)
    print("-" * 50)

Query: What is a retriever in LangChain?
--------------------------------------------------
Result 1:
LangChain is a framework for building applications with large language models.
Metadata: {'topic': 'langchain', 'source': 'langchain_intro'}
--------------------------------------------------
Result 2:
LangChain is a framework for building applications with large language models.
Metadata: {'topic': 'langchain', 'source': 'langchain_intro'}
--------------------------------------------------
